# XGBoost 再学習ノートブック（Stage 3 完了後）

新モデルが旧モデルより精度低下したため、原因を切り分ける。

| ステップ | 内容 | 目的 |
|----------|------|------|
| セル1〜3 | セットアップ・学習データ生成 | 共通前処理 |
| セル4A | **旧58特徴量** × **新データ** で再学習 | データ品質改善の効果だけを測る |
| セル4B | 新特徴量を1つずつ追加 | 効く特徴量だけ残す |
| セル4C | Optuna ハイパーパラメータ探索 | データ分布変化に合わせた再チューニング |
| セル5 | キャリブレーター再学習 | 採用モデルに合わせる |
| セル6 | 統合テスト | cal_prob合計・重要度確認 |

In [ ]:
# ── セル1: セットアップ ───────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

import os, sys
BASE_DIR = '/content/drive/MyDrive/keiba_ai'
sys.path.insert(0, BASE_DIR)
print(f'✅ BASE_DIR: {BASE_DIR}')

In [ ]:
# ── セル2: src/ 強制アップデート ──────────────────────────────
import urllib.request, time as _time
BASE_URL = 'https://raw.githubusercontent.com/hanagenuku/keiba_ai/main'
_cb = int(_time.time())  # キャッシュバスター（GitHub CDN対策）
files = [
    'src/tools/__init__.py', 'src/tools/tune_weights.py',
    'src/tools/calibrate.py', 'src/tools/analyze_divergence.py',
    'src/tools/rescrape_history.py', 'src/tools/build_training_data.py',
    'src/tools/train_xgb.py', 'src/tools/calibrate_xgb.py',
    'src/features/engine.py',
    'src/utils/config.py', 'src/utils/db.py', 'src/utils/model_registry.py',
    'src/scraper/parser.py', 'src/scraper/jra_scraper.py',
    'src/models/__init__.py', 'src/models/calibration.py',
    'src/models/calibration_xgb.py', 'src/models/predict.py',
    'src/betting/__init__.py', 'src/betting/make_bets.py',
    'src/betting/ev_filter.py', 'src/betting/app_json.py',
]
for rel in files:
    dest = f'{BASE_DIR}/{rel}'
    os.makedirs(os.path.dirname(dest), exist_ok=True)
    urllib.request.urlretrieve(f'{BASE_URL}/{rel}?nocache={_cb}', dest)
    print(f'OK {rel}')
for key in list(sys.modules.keys()):
    if 'src' in key:
        del sys.modules[key]

# ── 新特徴量が含まれているか確認 ──
with open(f'{BASE_DIR}/src/features/engine.py') as _f:
    _eng_src = _f.read()
for _kw in ['f_sex', 'f_age', 'f_track_cond', 'f_class_level']:
    _ok = _kw in _eng_src
    print(f'  {"✅" if _ok else "❌"} engine.py に {_kw} {"あり" if _ok else "なし！← 要確認"}')
print('done')

In [ ]:
# ── セル3: 学習データ生成（horse_features.csv）────────────────
# 約10〜20分かかる。既存CSVは horse_features_old.csv に自動バックアップ。

from src.tools.build_training_data import build_training_data
df_all = build_training_data(BASE_DIR)

# 新特徴量の充足率確認
print(f'\n── 新特徴量確認 ──')
NEW_FEAT_COLS = [
    'f_sex', 'f_age', 'f_track_cond', 'f_heavy_track_rate',
    'f_class_level', 'f_class_jump', 'f_finish_time_avg', 'f_time_diff_avg',
]
for c in NEW_FEAT_COLS:
    if c in df_all.columns:
        pct = 100 * df_all[c].notna().sum() / len(df_all)
        print(f'  ✅ {c}: {pct:.1f}%')
    else:
        print(f'  ❌ {c}: 列なし')

print(f'\n総列数: {len(df_all.columns)}  総行数: {len(df_all)}')

## ステップA: データ品質改善の効果を測る
旧58特徴量 × 新データで再学習。AUCが上がれば「データ品質改善は効いている」。

In [ ]:
# ── セル4A: 旧58特徴量 × 新データで再学習 ────────────────────
import pandas as pd
import numpy as np
import pickle, json, shutil
from xgboost import XGBClassifier
from sklearn.metrics import roc_auc_score, brier_score_loss

# 新特徴量列（相対化含む）をすべて除外
_NEW_ABSOLUTE = [
    'f_sex', 'f_age', 'f_track_cond', 'f_heavy_track_rate',
    'f_class_level', 'f_class_jump', 'f_finish_time_avg', 'f_time_diff_avg',
]
_NEW_RELATIVE = [
    'cl_f_heavy_track_rank', 'cl_f_heavy_track_vs_field',
    'cl_f_weight_load_rank', 'cl_f_weight_load_vs_field',
    'rl_f_finish_time_rank', 'rl_f_finish_time_vs_field',
    'rl_f_time_diff_rank',   'rl_f_time_diff_vs_field',
]
_EXCLUDE = {'race_id','date','horse_name','horse_num','place','is_fukusho'}
_EXCLUDE |= set(_NEW_ABSOLUTE) | set(_NEW_RELATIVE)

df = df_all.copy()
df['date_obj'] = pd.to_datetime(
    df['date'].astype(str).str.replace('-','',regex=False).str[:8],
    format='%Y%m%d', errors='coerce'
)
df = df.dropna(subset=['date_obj'])

TRAIN_END  = '2026-03-31'
VAL_START  = '2026-04-01'
VAL_END    = '2026-05-31'

train_df = df[df['date_obj'] <= pd.Timestamp(TRAIN_END)]
val_df   = df[(df['date_obj'] >= pd.Timestamp(VAL_START)) &
              (df['date_obj'] <= pd.Timestamp(VAL_END))]

feat_cols_A = [c for c in df.columns
               if c not in _EXCLUDE and c != 'date_obj'
               and df[c].dtype in ('float64','int64','float32','int32')]
print(f'ステップA 特徴量数: {len(feat_cols_A)}  (旧58と同じはずです)')
print(f'Train: {len(train_df)}行  Val: {len(val_df)}行')

X_tr = train_df[feat_cols_A].fillna(5.0)
y_tr = train_df['is_fukusho']
X_vl = val_df[feat_cols_A].fillna(5.0)
y_vl = val_df['is_fukusho']

pos_rate = y_tr.mean()
spw = round((1 - pos_rate) / max(pos_rate, 0.01), 2)

model_A = XGBClassifier(
    n_estimators=500, max_depth=6, learning_rate=0.05,
    subsample=0.8, colsample_bytree=0.8,
    min_child_weight=10, reg_alpha=0.1, reg_lambda=1.0,
    scale_pos_weight=spw, eval_metric='logloss',
    early_stopping_rounds=50, random_state=42, n_jobs=-1,
)
model_A.fit(X_tr, y_tr, eval_set=[(X_vl, y_vl)], verbose=100)

prob_A = model_A.predict_proba(X_vl)[:, 1]
auc_A  = roc_auc_score(y_vl, prob_A)
brier_A = brier_score_loss(y_vl, prob_A)

# 旧モデルで同じValデータを評価（比較用）
# ※旧モデルが現在のengine.pyにない特徴量(f_pace_match等)を使っている場合はスキップ
old_path = f'{BASE_DIR}/data/xgb_fukusho_model_old.pkl'
old_cols_path_check = f'{BASE_DIR}/data/xgb_feature_cols.json'
auc_old = brier_old = None
if os.path.exists(old_path) and os.path.exists(old_cols_path_check):
    try:
        with open(old_path,'rb') as f: old_model = pickle.load(f)
        with open(old_cols_path_check) as f: old_info = json.load(f)
        old_cols = old_info.get('feature_cols', [])
        # 現在のデータに存在しない列があれば比較不可
        missing = [c for c in old_cols if c not in df.columns]
        if missing:
            print(f'⚠ 旧モデルの特徴量 {len(missing)}列が現データにありません: {missing[:5]}...')
            print(f'  → 旧モデルとの直接比較はスキップします（特徴量構成が変わっているため）')
        else:
            X_old = val_df.reindex(columns=old_cols, fill_value=5.0)
            # XGBoostの特徴量名検証を一時的に無効化
            old_model.get_booster().feature_names = None
            prob_old = old_model.predict_proba(X_old.values)[:, 1]
            auc_old   = roc_auc_score(y_vl, prob_old)
            brier_old = brier_score_loss(y_vl, prob_old)
    except Exception as e:
        print(f'⚠ 旧モデル評価スキップ: {e}')

print(f'\n── ステップA 結果 ──')
if auc_old:
    print(f'  旧モデル(旧データ): AUC={auc_old:.4f}  Brier={brier_old:.4f}')
else:
    print(f'  旧モデル: 比較不可（特徴量構成が異なる）')
print(f'  ステップA(新データ): AUC={auc_A:.4f}  Brier={brier_A:.4f}')
if auc_old:
    delta = auc_A - auc_old
    print(f'  ΔAUC = {delta:+.4f}  → {"✅ データ品質改善が効いている" if delta > 0 else "⚠ データ品質改善のみでは改善なし"}')

# ステップAモデルを保存（後続ステップで使用）
with open(f'{BASE_DIR}/data/xgb_step_a.pkl','wb') as f: pickle.dump(model_A, f)
with open(f'{BASE_DIR}/data/xgb_step_a_cols.json','w') as f:
    json.dump({'feature_cols': feat_cols_A}, f)
print(f'\n保存: xgb_step_a.pkl')

## ステップB: 新特徴量を1つずつ追加
ステップAをベースに新特徴量を追加して効果を測る。AUCが下がる特徴量は除外。

In [ ]:
# ── セル4B: 新特徴量を1つずつ追加して効果測定 ────────────────
# 各特徴量の単独追加効果を測定する（相対特徴量はペアで追加）

_FEAT_CANDIDATES = [
    # (表示名, [追加する列リスト])
    ('f_sex',              ['f_sex']),
    ('f_age',              ['f_age']),
    ('f_track_cond',       ['f_track_cond']),
    ('f_heavy_track_rate', ['f_heavy_track_rate',
                            'cl_f_heavy_track_rank','cl_f_heavy_track_vs_field']),
    ('f_class_level',      ['f_class_level']),
    ('f_class_jump',       ['f_class_jump']),
    ('f_finish_time_avg',  ['f_finish_time_avg',
                            'rl_f_finish_time_rank','rl_f_finish_time_vs_field']),
    ('f_time_diff_avg',    ['f_time_diff_avg',
                            'rl_f_time_diff_rank','rl_f_time_diff_vs_field']),
    ('f_weight_load_rel',  ['cl_f_weight_load_rank','cl_f_weight_load_vs_field']),
]

results_B = []
base_auc = auc_A

for name, add_cols in _FEAT_CANDIDATES:
    # dfに存在する列だけ追加
    valid_adds = [c for c in add_cols if c in df_all.columns]
    if not valid_adds:
        print(f'  ⚠ {name}: 列なし（スキップ）')
        continue

    cols_try = feat_cols_A + valid_adds
    X_tr_b = train_df[cols_try].fillna(5.0)
    X_vl_b = val_df[cols_try].fillna(5.0)

    m = XGBClassifier(
        n_estimators=300, max_depth=6, learning_rate=0.05,
        subsample=0.8, colsample_bytree=0.8,
        min_child_weight=10, reg_alpha=0.1, reg_lambda=1.0,
        scale_pos_weight=spw, eval_metric='logloss',
        early_stopping_rounds=30, random_state=42, n_jobs=-1,
    )
    m.fit(X_tr_b, y_tr, eval_set=[(X_vl_b, y_vl)], verbose=False)
    p = m.predict_proba(X_vl_b)[:, 1]
    auc_b = roc_auc_score(y_vl, p)
    delta = auc_b - base_auc
    adopt = delta >= 0
    results_B.append({'name': name, 'cols': valid_adds, 'auc': auc_b, 'delta': delta, 'adopt': adopt})
    mark = '✅採用' if adopt else '❌除外'
    print(f'  {mark}  {name:<22} AUC={auc_b:.4f}  ΔAUC={delta:+.4f}')

# 採用特徴量でまとめて再学習
adopted_cols = []
for r in results_B:
    if r['adopt']:
        adopted_cols.extend(r['cols'])

feat_cols_B = feat_cols_A + adopted_cols
print(f'\n採用後の特徴量数: {len(feat_cols_B)}')

X_tr_b = train_df[feat_cols_B].fillna(5.0)
X_vl_b = val_df[feat_cols_B].fillna(5.0)

model_B = XGBClassifier(
    n_estimators=500, max_depth=6, learning_rate=0.05,
    subsample=0.8, colsample_bytree=0.8,
    min_child_weight=10, reg_alpha=0.1, reg_lambda=1.0,
    scale_pos_weight=spw, eval_metric='logloss',
    early_stopping_rounds=50, random_state=42, n_jobs=-1,
)
model_B.fit(X_tr_b, y_tr, eval_set=[(X_vl_b, y_vl)], verbose=100)
prob_B = model_B.predict_proba(X_vl_b)[:, 1]
auc_B  = roc_auc_score(y_vl, prob_B)
brier_B = brier_score_loss(y_vl, prob_B)

print(f'\n── ステップB 結果 ──')
print(f'  ステップA(ベース): AUC={auc_A:.4f}')
print(f'  ステップB(採用後): AUC={auc_B:.4f}  Brier={brier_B:.4f}')

with open(f'{BASE_DIR}/data/xgb_step_b.pkl','wb') as f: pickle.dump(model_B, f)
with open(f'{BASE_DIR}/data/xgb_step_b_cols.json','w') as f:
    json.dump({'feature_cols': feat_cols_B}, f)
print(f'保存: xgb_step_b.pkl')

## ステップC: Optuna ハイパーパラメータ探索
ステップBの特徴量セットを固定して max_depth / learning_rate / min_child_weight を探索。

In [ ]:
# ── セル4C: Optuna ハイパーパラメータ探索 ─────────────────────
# ステップBの特徴量セットを固定して探索
# 合格基準: AUC ≥ 0.75、Brier ≤ 0.165（ステップAを新ベースライン）

try:
    import optuna
except ImportError:
    import subprocess
    subprocess.run(['pip', 'install', 'optuna', '-q'])
    import optuna

optuna.logging.set_verbosity(optuna.logging.WARNING)

X_tr_c = train_df[feat_cols_B].fillna(5.0)
X_vl_c = val_df[feat_cols_B].fillna(5.0)

def objective(trial):
    params = {
        'n_estimators':     trial.suggest_int('n_estimators', 200, 600),
        'max_depth':        trial.suggest_int('max_depth', 4, 8),
        'learning_rate':    trial.suggest_float('learning_rate', 0.01, 0.1, log=True),
        'subsample':        trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'min_child_weight': trial.suggest_int('min_child_weight', 5, 30),
        'reg_alpha':        trial.suggest_float('reg_alpha', 0.0, 1.0),
        'reg_lambda':       trial.suggest_float('reg_lambda', 0.5, 5.0),
        'scale_pos_weight': spw,
        'eval_metric':      'logloss',
        'early_stopping_rounds': 30,
        'random_state':     42,
        'n_jobs':           -1,
    }
    m = XGBClassifier(**params)
    m.fit(X_tr_c, y_tr, eval_set=[(X_vl_c, y_vl)], verbose=False)
    p = m.predict_proba(X_vl_c)[:, 1]
    return roc_auc_score(y_vl, p)

N_TRIALS = 30  # ★ 時間に余裕があれば50〜100に増やす
print(f'Optuna 探索開始 ({N_TRIALS}試行)...')
study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=N_TRIALS, show_progress_bar=True)

best = study.best_params
print(f'\n── 最良パラメータ ──')
for k, v in best.items():
    print(f'  {k}: {v}')
print(f'  Best AUC: {study.best_value:.4f}')

# 最良パラメータで最終モデルを学習
model_C = XGBClassifier(
    **best,
    scale_pos_weight=spw,
    eval_metric='logloss',
    early_stopping_rounds=50,
    random_state=42,
    n_jobs=-1,
)
model_C.fit(X_tr_c, y_tr, eval_set=[(X_vl_c, y_vl)], verbose=50)
prob_C = model_C.predict_proba(X_vl_c)[:, 1]
auc_C   = roc_auc_score(y_vl, prob_C)
brier_C = brier_score_loss(y_vl, prob_C)

# ── 3ステップ比較 ──
AUC_THRESHOLD   = 0.75
BRIER_THRESHOLD = 0.165

print(f'\n══ ステップ比較（新ベースライン = ステップA） ══')
print(f'  A(新データ・旧特徴): AUC={auc_A:.4f}  Brier={brier_A:.4f}  ← 新ベースライン')
print(f'  B(新特徴量採用後) : AUC={auc_B:.4f}  Brier={brier_B:.4f}  ΔAUC={auc_B-auc_A:+.4f}')
print(f'  C(HP最適化後)    : AUC={auc_C:.4f}  Brier={brier_C:.4f}  ΔAUC={auc_C-auc_A:+.4f}')

print(f'\n── 合格基準チェック（AUC≥{AUC_THRESHOLD} / Brier≤{BRIER_THRESHOLD}） ──')
for label, auc_v, brier_v in [('A', auc_A, brier_A), ('B', auc_B, brier_B), ('C', auc_C, brier_C)]:
    ok = auc_v >= AUC_THRESHOLD and brier_v <= BRIER_THRESHOLD
    print(f'  ステップ{label}: {"✅ 合格" if ok else "⚠ 未達"}  AUC={auc_v:.4f}  Brier={brier_v:.4f}')

# ── ステップA/B/C のうち最良（AUC最大）を採用 ──
best_auc = max(auc_A, auc_B, auc_C)
step_name, best_model, best_cols = {
    auc_A: ('A', model_A, feat_cols_A),
    auc_B: ('B', model_B, feat_cols_B),
    auc_C: ('C', model_C, feat_cols_B),
}[best_auc]

old_model_path = f'{BASE_DIR}/data/xgb_fukusho_model.pkl'
old_cols_path  = f'{BASE_DIR}/data/xgb_feature_cols.json'

if os.path.exists(old_model_path):
    shutil.copy2(old_model_path, f'{BASE_DIR}/data/xgb_fukusho_model_pre_stage3.pkl')

with open(old_model_path, 'wb') as f: pickle.dump(best_model, f)
with open(old_cols_path, 'w') as f:
    json.dump({'feature_cols': best_cols, 'step': step_name,
               'auc': round(best_auc, 4)}, f, indent=2)

FINAL_MODEL = best_model
FINAL_COLS  = best_cols
print(f'\n✅ ステップ{step_name}モデルを正式採用 (AUC={best_auc:.4f})')

In [ ]:
# ── セル5: XGBキャリブレーター再学習 ─────────────────────────
# セル4Cで正式採用したモデルに合わせて再学習
for key in list(sys.modules.keys()):
    if 'src' in key:
        del sys.modules[key]

from src.tools.calibrate_xgb import run_xgb_calibration
run_xgb_calibration(BASE_DIR, test_days=60, n_bins=20)

In [ ]:
# ── セル6: 統合テスト ──────────────────────────────────────────
import pickle, json, numpy as np, pandas as pd

model_path = f'{BASE_DIR}/data/xgb_fukusho_model.pkl'
cols_path  = f'{BASE_DIR}/data/xgb_feature_cols.json'
calib_path = f'{BASE_DIR}/data/xgb_calibrator.pkl'

with open(model_path, 'rb') as f: model = pickle.load(f)
with open(cols_path)         as f: cols  = json.load(f)['feature_cols']
with open(calib_path, 'rb')  as f: calib = pickle.load(f)

print(f'✅ モデル: {model_path}')
print(f'✅ 特徴量数: {len(cols)}')
print(f'✅ キャリブレーター: {type(calib).__name__}')
print(f'   利用可能メソッド: {[m for m in dir(calib) if not m.startswith("_")]}')

# IsotonicCalibrator は .transform() を使う
dummy = pd.DataFrame([{c: np.random.normal(5.0, 1.5) for c in cols} for _ in range(16)])
raw_probs = model.predict_proba(dummy)[:, 1]
cal_probs = np.array(calib.transform(raw_probs))

print(f'\n── ダミー16頭 ──')
print(f'  raw_prob range: {raw_probs.min():.3f} ~ {raw_probs.max():.3f}')
print(f'  cal_prob 合計: {cal_probs.sum():.3f}  (2.8〜3.2 が正常)')
print(f'  cal_prob 分散: {raw_probs.max() - raw_probs.min():.3f}  (>0.3 が正常)')
print(f'  cal_prob合計  {"✅" if 2.8 <= cal_probs.sum() <= 3.2 else "⚠"}')
print(f'  分散          {"✅" if raw_probs.max() - raw_probs.min() > 0.3 else "⚠"}')

# 特徴量重要度トップ20
imps = sorted(zip(cols, model.feature_importances_), key=lambda x: x[1], reverse=True)[:20]
print('\n── 特徴量重要度 Top 20 ──')
_new = {'f_sex','f_age','f_track_cond','f_heavy_track_rate','f_class_level',
        'f_class_jump','f_finish_time_avg','f_time_diff_avg'}
for name, imp in imps:
    mark = '★新' if name in _new else ('★' if name in ('f_cl_rank','f_rl_rank') else '')
    print(f'  {name:<40} {imp*100:.2f}% {mark}')